# Wrocław Smog Analysis — Zbieranie danych

**Co robi ten notebook:**  
Pobiera dane o jakości powietrza z publicznego API GIOŚ (Główny Inspektorat Ochrony Środowiska) dla wszystkich stacji pomiarowych we Wrocławiu.

**Pipeline:**
```
API GIOŚ
├── stacje Wrocławia     → stations.csv
├── czujniki per stacja  → sensors.csv
└── pomiary per czujnik  → measurements.csv  ← surowe dane
```

**Dlaczego GIOŚ?**  
Publiczne API rządowe, bez rejestracji, dane od 2015 roku, format JSON. Idealne do pokazania pipeline'u DE.

## 1. Importy i konfiguracja

- `requests` — wysyła zapytania HTTP do API  
- `pandas` — tabele danych  
- `time.sleep` — pauza między zapytaniami (grzeczny klient API)  
- `BASE_URL` — adres bazowy API, wersja 1 (v1 jest aktywna od 2025)

In [ ]:
import requests
import pandas as pd
from time import sleep

BASE_URL = "https://api.gios.gov.pl/pjp-api/v1/rest"

# Wszystkie dane zapisujemy tutaj
DATA_DIR = "../data"

print("Gotowe.")

## 2. Pobieramy stacje pomiarowe we Wrocławiu

API zwraca stacje z całej Polski — musimy odfiltrować tylko Wrocław.

**Jak działa API z paginacją:**  
Wyobraź sobie książkę telefoniczną podzieloną na strony. API nie wyrzuca wszystkich danych naraz — daje je po kawałku (`size=100` wierszy na raz). Pętla `while` przełącza strony dopóki nie pobierzemy wszystkich.

To bardzo częsty pattern w pracy z API — warto go znać na pamięć.

In [ ]:
def get_all_stations():
    """Pobiera wszystkie stacje z API (paginacja)."""
    all_stations = []
    page = 0

    while True:
        url = f"{BASE_URL}/station/findAll"
        params = {"page": page, "size": 100}
        response = requests.get(url, params=params)
        data = response.json()

        # API zwraca dane pod kluczem 'Lista stacji pomiarowych'
        # Sprawdzamy jaki klucz ma odpowiedź
        stations = data.get("Lista stacji pomiarowych", [])

        if not stations:
            break  # Brak danych = dotarliśmy do końca

        all_stations.extend(stations)
        page += 1
        sleep(0.5)

    return all_stations


print("Pobieram wszystkie stacje...")
all_stations = get_all_stations()
print(f"Łącznie stacji w Polsce: {len(all_stations)}")

# Podgląd struktury jednej stacji — żeby wiedzieć jakie ma pola
print("\nStruktura jednej stacji:")
print(all_stations[0] if all_stations else "Brak danych")

## 3. Filtrujemy tylko Wrocław

List comprehension — przefiltruj listę i zostaw tylko stacje z Wrocławia.  
Sprawdzamy pole `city.name` — pola zagnieżdżone w JSON w pandas obsługujemy przez `.get()`.

In [ ]:
wroclaw_stations = [
    s for s in all_stations
    if s.get("city", {}).get("name", "") == "Wrocław"
]

print(f"Stacje we Wrocławiu: {len(wroclaw_stations)}")
print()

for s in wroclaw_stations:
    print(f"  ID {s['id']:>5} | {s['stationName']}")
    print(f"           lat: {s['gegrLat']}, lon: {s['gegrLon']}")

## 4. Zapisujemy stacje do CSV

Wyciągamy tylko pola które nas interesują — spłaszczamy zagnieżdżony JSON do płaskiej tabeli.  
To klasyczny krok w DE: **normalizacja** — z JSON (hierarchia) do tabeli (relacja).

In [ ]:
stations_df = pd.DataFrame([{
    "station_id":   s["id"],
    "name":         s["stationName"],
    "lat":          s["gegrLat"],
    "lon":          s["gegrLon"],
    "address":      s.get("addressStreet", ""),
    "city":         s["city"]["name"]
} for s in wroclaw_stations])

stations_df.to_csv(f"{DATA_DIR}/stations.csv", index=False)

print("Zapisano stations.csv")
stations_df

## 5. Pobieramy czujniki dla każdej stacji

Każda stacja ma kilka czujników — każdy mierzy inny parametr: PM2.5, PM10, NO2, O3, CO...  
Teraz dla każdej z 5 stacji pytamy API: "jakie masz czujniki?"

**Zagnieżdżona pętla** — dla każdej stacji, pobierz jej czujniki.  
W DE to się nazywa **"fan-out"** — z 1 stacji rozchodzisz się na N czujników.

In [ ]:
all_sensors = []

for _, station in stations_df.iterrows():
    url = f"{BASE_URL}/station/sensors/{station['station_id']}"
    response = requests.get(url)
    sensors = response.json().get("Lista sensorów", [])

    for sensor in sensors:
        all_sensors.append({
            "sensor_id":    sensor["id"],
            "station_id":   station["station_id"],
            "station_name": station["name"],
            "param_name":   sensor["param"]["paramName"],
            "param_code":   sensor["param"]["paramCode"],
        })

    print(f"  Stacja {station['station_id']}: {len(sensors)} czujników")
    sleep(0.5)

sensors_df = pd.DataFrame(all_sensors)
sensors_df.to_csv(f"{DATA_DIR}/sensors.csv", index=False)

print(f"\nŁącznie czujników: {len(sensors_df)}")
sensors_df

## 6. Pobieramy pomiary

To jest serce projektu — dla każdego czujnika pobieramy historyczne wartości.

API zwraca listę par `{date, value}`. Każda to jeden pomiar (zazwyczaj godzinowy).  
Dorzucamy `sensor_id` i `param_code` żeby potem wiedzieć skąd pochodzi każdy wiersz.

**Uwaga:** To zapytanie zajmuje chwilę — pobieramy dane dla wszystkich czujników. `sleep(0.5)` jest tu szczególnie ważny.

In [ ]:
all_measurements = []

for _, sensor in sensors_df.iterrows():
    url = f"{BASE_URL}/data/getData/{sensor['sensor_id']}"
    response = requests.get(url)
    data = response.json()

    values = data.get("Lista danych pomiarowych", [])

    for v in values:
        if v["value"] is None:
            continue  # Pomijamy puste pomiary
        all_measurements.append({
            "sensor_id":  sensor["sensor_id"],
            "station_id": sensor["station_id"],
            "param_code": sensor["param_code"],
            "date":       v["date"],
            "value":      v["value"],
        })

    print(f"  Czujnik {sensor['sensor_id']} ({sensor['param_code']}): {len(values)} pomiarów")
    sleep(0.5)

measurements_df = pd.DataFrame(all_measurements)
measurements_df.to_csv(f"{DATA_DIR}/measurements.csv", index=False)

print(f"\nŁącznie pomiarów: {len(measurements_df):,}")
measurements_df.head(10)

## Podsumowanie

Po uruchomieniu tego notebooka w folderze `data/` masz 3 pliki:

| Plik | Co zawiera |
|---|---|
| `stations.csv` | 5 stacji Wrocławia z ID i współrzędnymi GPS |
| `sensors.csv` | Czujniki per stacja (PM2.5, PM10, NO2, ...) |
| `measurements.csv` | Historyczne pomiary — to są nasze surowe dane |

**Następny krok:** `02_clean.ipynb` — czyszczenie i ładowanie do SQLite.